In [27]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
#to clean the memory

In [28]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
!pip install sentence-transformers faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
!pip install -q bitsandbytes accelerate
!pip install -U bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [31]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

#a 4-bit quantization configuration (for optimizing memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

def generate_text(prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.2,
        pad_token_id=tokenizer.eos_token_id
    )
    
    generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [33]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
#define the response schema
response_schemas = [
    ResponseSchema(
        name="dish_name", 
        description="The overall dish name or meal description provided by the user.", 
        type="string"
    ),
    ResponseSchema(
        name="components", 
        description="A list of extracted ingredients/components. Each item must contain 'name' (string) and 'estimated_weight_grams' (integer/float representing portion size in grams).", 
        type="list"
    )
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

In [34]:
print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"dish_name": string  // The overall dish name or meal description provided by the user.
	"components": list  // A list of extracted ingredients/components. Each item must contain 'name' (string) and 'estimated_weight_grams' (integer/float representing portion size in grams).
}
```


In [ ]:
import re
from langchain_core.prompts import PromptTemplate



#setup prompt Template
meal_parser_template = """You are an expert nutritionist AI assistant.
Your task is to analyze user text input describing a meal, break it down into its core components/ingredients, and estimate the weight in grams for each item based on mentioned portion sizes or standard default serving sizes.

{format_instructions}

Important Guidelines:
- If the user specifies a quantity of items (e.g., "2 boiled eggs"), multiply standard unit weights accordingly (e.g., 1 large egg ≈ 50g, so 2 eggs = 100g).
- If the user does not explicitly mention exact weights, infer realistic default portion sizes in grams.
- Return ONLY a valid JSON code snippet wrapped in markdown ```json ``` block.
- Use explicit whole-food ingredient names (e.g., "boiled egg", "skinless boneless grilled chicken breast", "apple").

User Input: {user_input}
"""

prompt_template = PromptTemplate(
    template=meal_parser_template,
    input_variables=["user_input", "format_instructions"]
)

# #format prompt_template into a prompt string
# prompt = prompt_template.format(
#     user_input=user_meal,
#     format_instructions=format_instructions
# )

# response = generate_text(prompt)
# print(response)

In [36]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return f"```json\n{matches[-1].strip()}\n```"
    return text

In [37]:
# json_text = extract_json_block(response)
# output_data = output_parser.parse(json_text)
# print(output_data)

In [38]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

#load the dataset
df_foods = pd.read_csv("/kaggle/input/datasets/ahmedbayomy/food-data/comprehensive_foods_usda.csv")

#get key columns
df_foods["food_name"] = df_foods["food_name"].fillna("Unknown")
df_foods["calories"] = df_foods["calories"].fillna(0.0)
df_foods["protein_g"] = df_foods["protein_g"].fillna(0.0)
df_foods["carbs_g"] = df_foods["carbs_g"].fillna(0.0)
df_foods["fat_g"] = df_foods["fat_g"].fillna(0.0)

#get the embeddings
food_names = df_foods["food_name"].tolist()

embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
food_embeddings = embed_model.encode(food_names, convert_to_numpy=True)

#create Faiss L2 Index
dimension = food_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(food_embeddings)

print(f"Successfully indexed {index.ntotal} items in FAISS.")

Successfully indexed 40000 items in FAISS.


In [39]:
def calculate_calories_and_macros(parsed_data):
    dish_name = parsed_data.get("dish_name", "Meal")
    components = parsed_data.get("components", [])
    
    breakdown = []
    total_calories = 0.0
    total_protein = 0.0
    total_carbs = 0.0
    total_fat = 0.0
    
    composite_words = ['sandwich', 'burger', 'pizza', 'pasta', 'fettuccine', 'noodle', 
                       'soup', 'stew', 'babyfood', 'platter', 'salad with', 'appetizer', 'dressing']
    
    for comp in components:
        ingredient_name = comp["name"]
        weight_grams = float(comp["estimated_weight_grams"])
        
        query_vector = embed_model.encode([ingredient_name], convert_to_numpy=True)
        distances, indices = index.search(query_vector, k=15)
        
        query_lower = ingredient_name.lower()
        is_composite_query = any(w in query_lower for w in composite_words)
        
        matched_row = None
        
        #priority 1: Search for SR Legacy / Foundation standard foods (unbranded whole foods)
        for candidate_idx in indices[0]:
            row = df_foods.iloc[candidate_idx]
            food_title = str(row["food_name"]).lower()
            dtype = str(row.get("data_type", ""))
            
            if not is_composite_query and any(w in food_title for w in composite_words):
                continue
                
            if dtype in ['SR Legacy', 'Foundation']:
                matched_row = row
                break
                
        #priority 2: Fallback to best non-composite branded item if no SR Legacy item found
        if matched_row is None:
            for candidate_idx in indices[0]:
                row = df_foods.iloc[candidate_idx]
                food_title = str(row["food_name"]).lower()
                if not is_composite_query and any(w in food_title for w in composite_words):
                    continue
                matched_row = row
                break
                
        #priority 3: Fallback to top match
        if matched_row is None:
            matched_row = df_foods.iloc[indices[0][0]]
        
        multiplier = weight_grams / 100.0
        
        cals = round(matched_row["calories"] * multiplier, 2)
        protein = round(matched_row["protein_g"] * multiplier, 2)
        carbs = round(matched_row["carbs_g"] * multiplier, 2)
        fat = round(matched_row["fat_g"] * multiplier, 2)
        
        total_calories += cals
        total_protein += protein
        total_carbs += carbs
        total_fat += fat
        
        breakdown.append({
            "ingredient": ingredient_name,
            "matched_db_food": matched_row["food_name"],
            "weight_grams": weight_grams,
            "calories": cals,
            "protein_g": protein,
            "carbs_g": carbs,
            "fat_g": fat
        })
        
    return {
        "dish_name": dish_name,
        "grand_totals": {
            "total_calories": round(total_calories, 2),
            "total_protein_g": round(total_protein, 2),
            "total_carbs_g": round(total_carbs, 2),
            "total_fat_g": round(total_fat, 2)
        },
        "itemized_breakdown": breakdown
    }

In [40]:
def print_meal_summary(report):
    breakdown = report['itemized_breakdown']
    
    #find the required column width (min is 18)
    max_ing_len = max([len(item['ingredient']) for item in breakdown] + [18])
    col_width = max_ing_len + 2  #padding space
    
    table_width = col_width + 48  #total table width
    
    print("=" * table_width)
    print(f" 🍽  MEAL NUTRITION SUMMARY: {report['dish_name'].upper()}")
    print("=" * table_width)
    
    #table header
    print(f"{'Ingredient':<{col_width}} | {'Portion':<8} | {'Calories':<10} | {'Protein':<8} | {'Carbs':<8} | {'Fat':<8}")
    print("-" * table_width)
    
    for item in breakdown:
        ing = item['ingredient']  # Full name, no slicing!
        portion = f"{int(item['weight_grams'])}g"
        cals = f"{item['calories']} kcal"
        protein = f"{item['protein_g']}g"
        carbs = f"{item['carbs_g']}g"
        fat = f"{item['fat_g']}g"
        print(f"{ing:<{col_width}} | {portion:<8} | {cals:<10} | {protein:<8} | {carbs:<8} | {fat:<8}")
    
    totals = report['grand_totals']
    print("=" * table_width)
    print(" 📊 TOTAL NUTRITION")
    print("-" * table_width)
    print(f" 🔥 Calories : {totals['total_calories']} kcal")
    print(f" 🥩 Protein  : {totals['total_protein_g']} g")
    print(f" 🍞 Carbs    : {totals['total_carbs_g']} g")
    print(f" 🥑 Fat      : {totals['total_fat_g']} g")
    print("=" * table_width)



In [41]:
from langchain_core.runnables import RunnableLambda

parse_chain = (
    RunnableLambda(lambda user_input: prompt_template.format(user_input=user_input, format_instructions=format_instructions))
    | RunnableLambda(generate_text)
    | RunnableLambda(extract_json_block)
    | RunnableLambda(output_parser.parse)
)


api_chain = parse_chain | RunnableLambda(calculate_calories_and_macros)



# full_nutrition_chain = (
#     parse_chain
#     | RunnableLambda(calculate_calories_and_macros)
#     | RunnableLambda(print_meal_summary)
# )

# full_nutrition_chain.invoke("I had 2 boiled eggs, an apple, and 200g of grilled chicken breast")

In [ ]:
import re


def is_arabic(text: str) -> bool:
  return bool(re.search(r'[\u0600-\u06FF]', text))


def translate_ar_to_en(arabic_text: str) -> str:
  prompt = (
      "Translate the following Arabic meal description into clear English."
      " Output ONLY the translated English text with no additional text or"
      f" formatting.\n\nArabic: {arabic_text}\nEnglish:"
  )
  translated = generate_text(prompt, max_new_tokens=128).strip()
  return translated


def translate_batch_en_to_ar(english_texts: list[str]) -> list[str]:
  if not english_texts:
    return []

  items_str = "\n".join([f"{i+1}. {t}" for i, t in enumerate(english_texts)])
  prompt = f"""Translate the following list of food items from English to natural Arabic.
Maintain the exact numbered list format. Output ONLY the numbered Arabic translations.

{items_str}

Arabic:"""

  raw_response = generate_text(prompt, max_new_tokens=256).strip()

  #parse numbered response back into a list
  translated_lines = []
  for line in raw_response.split("\n"):
    cleaned = re.sub(r"^\d+[\.\)]\s*", "", line).strip()
    if cleaned:
      translated_lines.append(cleaned)

  #fallback in case line counts don't match
  if len(translated_lines) == len(english_texts):
    return translated_lines

  #fallback in case list parsing differed
  fallback_list = []
  for text in english_texts:
    p = f"Translate this food item to Arabic. Output ONLY the Arabic text: {text}"
    fallback_list.append(generate_text(p, max_new_tokens=64).strip())
  return fallback_list

In [43]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import asyncio
import uvicorn
from fastapi import FastAPI
import nest_asyncio
from pydantic import BaseModel
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

nest_asyncio.apply()

app = FastAPI(title="AI Calorie Calculator API")


class MealRequest(BaseModel):
  user_input: str


@app.post("/calculate")
def calculate_meal(request: MealRequest):
  raw_input = request.user_input.strip()

  #detect if input is Arabic
  input_is_arabic = is_arabic(raw_input)

  # if Arabic, translate to English for FAISS & USDA matching
  if input_is_arabic:
    english_input = translate_ar_to_en(raw_input)
  else:
    english_input = raw_input

  #run the AI nutrition calculation chain in English
  result = api_chain.invoke(english_input)

  #if the input was Arabic, translate the result back to Arabic
  if input_is_arabic:
    texts_to_translate = [result.get("dish_name", "Meal")]
    for item in result["itemized_breakdown"]:
      texts_to_translate.append(item["ingredient"])
      texts_to_translate.append(item["matched_db_food"])

    #translating
    arabic_translations = translate_batch_en_to_ar(texts_to_translate)

    result["dish_name"] = arabic_translations[0]

    idx = 1
    for item in result["itemized_breakdown"]:
      item["ingredient"] = arabic_translations[idx]
      item["matched_db_food"] = arabic_translations[idx + 1]
      idx += 2

  return result


#==========================
#ngrok & Server startup
#=========================

user_secrets = UserSecretsClient()
NGROK_AUTHTOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

public_url = ngrok.connect(8000).public_url

print("=" * 60)
print(f"🚀 Public API URL: {public_url}")
print("=" * 60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

await server.serve()

🚀 Public API URL: https://flashbulb-geography-sanding.ngrok-free.dev


INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [48]
